# Complément GPU : vulnérabilités, pass@k, énergie mesurée, mémoire CPU

**À lancer par CHAQUE membre, pour SON modèle** (seule ligne à changer : `MODEL_KEY`). Mêmes conditions que le benchmark de base : Kaggle, GPU T4, bf16, `Settings > Internet > On`, session neuve.

Prérequis : `prompts_securite.json` et `config_complement.json` doivent être dans `benchmarking/` sur la branche clonée (fichiers COMMUNS à ajouter après accord du groupe).

| Partie | Ce qui est produit |
|---|---|
| 1 | `results/energie_<modele>.csv` : énergie GPU **mesurée** (NVML), puissance, mémoire CPU, pour les 15 prompts + 9 prompts sécurité |
| 1 | `results/vulnerabilites_<modele>.csv` : réponses aux 9 prompts de sécurité |
| 2 | `results/passk_<modele>.json` : pass@1, pass@5, pass@10 sur les 20 problèmes HumanEval |
| 3 | notation manuelle des vulnérabilités (TP/FP/FN/TN) |

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # un seul GPU -> mesures comparables

REPO_URL  = "https://github.com/FatiAllali123/SLM-Software-Engineering.git"
REPO_NAME = "SLM-Software-Engineering"
BRANCH    = "main"   # A CHANGER : branche qui contient prompts_securite.json et config_complement.json

if not os.path.exists("../config.json"):
    !git clone -b {BRANCH} {REPO_URL}
    %cd {REPO_NAME}/benchmarking/notebooks

os.makedirs("../results", exist_ok=True)

Cloning into 'SLM-Software-Engineering'...
remote: Enumerating objects: 157, done.
remote: Counting objects: 100% (157/157), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 157 (delta 64), reused 69 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (157/157), 118.11 KiB | 3.03 MiB/s, done.
Resolving deltas: 100% (64/64), done.
/kaggle/working/SLM-Software-Engineering/benchmarking/notebooks


In [2]:
!pip install -r ../requirements.txt -q
!pip install -q "transformers>=4.53" psutil nvidia-ml-py
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-be680e28-7e53-0526-e888-6051a03108a6)
GPU 1: Tesla T4 (UUID: GPU-c6d750c1-811c-c309-bc27-a318738885bf)


## Gemma uniquement
Acceptez les conditions du modèle sur Hugging Face, puis décommentez ces lignes (ne poussez jamais votre token) :

In [ ]:
# from huggingface_hub import login
# from kaggle_secrets import UserSecretsClient
# login(token=UserSecretsClient().get_secret("HF_TOKEN"))

In [3]:
import json, time, sys, re, threading
from math import comb
import torch, psutil, pynvml
import pandas as pd
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

sys.path.append("..")
from utils.safe_exec import run_tests
from utils.extract import extract_code

# === SEULE LIGNE A CHANGER : "qwen", "phi", "gemma" ou "smollm" ===
MODEL_KEY = "phi"
# ====================================================================

for f in ["../config.json", "../config_complement.json", "../prompts_securite.json", "../humaneval_subset.json"]:
    assert os.path.exists(f), f"{f} introuvable : ajoutez-le a la branche clonee (BRANCH) puis relancez."

config = json.load(open("../config.json"))
CC = json.load(open("../config_complement.json"))
SEC_PROMPTS = json.load(open("../prompts_securite.json", encoding="utf-8"))["prompts"]
PK, MON = CC["passk"], CC["monitor"]

MODEL_NAME = config["models"][MODEL_KEY]
GEN_PARAMS = config["generation_params"]
HE_CONFIG = config["humaneval"]
SYSTEM_PROMPT = config["no_think_models"].get(MODEL_KEY)
model_slug = MODEL_NAME.split("/")[-1].lower()
GPU_NAME = torch.cuda.get_device_name(0)
print(MODEL_NAME, "|", model_slug, "|", GPU_NAME, "| transformers", transformers.__version__)

microsoft/Phi-4-mini-instruct | phi-4-mini-instruct | Tesla T4 | transformers 5.16.1


## Outil de mesure (énergie GPU + mémoire CPU)
Un thread lit toutes les 50 ms la **puissance du GPU** (NVML) et la **mémoire RAM du processus** (RSS) pendant chaque génération.
- **energy_wh** = puissance moyenne × durée (énergie brute du GPU).
- **energy_net_wh** = energy_wh moins la consommation au repos (`idle`) sur la même durée.
- Limites à citer dans le rapport : NVML mesure le GPU seul (pas le CPU ni la machine) et se rafraîchit lentement (environ 100 ms).

In [4]:
pynvml.nvmlInit()
NVML_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0)
print("NVML GPU 0 :", pynvml.nvmlDeviceGetName(NVML_HANDLE))

def gpu_power_w():
    return pynvml.nvmlDeviceGetPowerUsage(NVML_HANDLE) / 1000.0

class Monitor:
    def __init__(self, power_fn=gpu_power_w, interval=MON["sample_interval_s"]):
        self.power_fn, self.interval = power_fn, interval
    def __enter__(self):
        self.t, self.p, self.rss, self._stop = [], [], [], False
        self._proc = psutil.Process()
        self._th = threading.Thread(target=self._loop, daemon=True)
        self._t0 = time.time(); self._th.start()
        return self
    def _loop(self):
        while not self._stop:
            try:
                self.p.append(self.power_fn()); self.t.append(time.time())
                self.rss.append(self._proc.memory_info().rss)
            except Exception:
                pass
            time.sleep(self.interval)
    def __exit__(self, *a):
        self.duration = time.time() - self._t0
        self._stop = True; self._th.join()
    @property
    def avg_power_w(self):
        if len(self.p) >= 2:
            area = sum((self.p[i] + self.p[i+1]) / 2 * (self.t[i+1] - self.t[i]) for i in range(len(self.p) - 1))
            span = self.t[-1] - self.t[0]
            return area / span if span > 0 else self.p[0]
        return self.p[0] if self.p else float("nan")
    @property
    def peak_power_w(self):
        return max(self.p) if self.p else float("nan")
    @property
    def cpu_rss_peak_go(self):
        return max(self.rss) / 1e9 if self.rss else float("nan")

NVML GPU 0 : Tesla T4


In [5]:
torch.manual_seed(PK["seed"])
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
DTYPE = torch.bfloat16 if config["precision"] == "bf16" else torch.float16
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", torch_dtype=DTYPE)
model.eval()
print("Precision :", config["precision"])

def build_inputs(user_prompt):
    messages = []
    if SYSTEM_PROMPT:
        messages.append({"role": "system", "content": SYSTEM_PROMPT})
    messages.append({"role": "user", "content": user_prompt})
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                         return_tensors="pt", return_dict=True).to(model.device)

def run_measured(user_prompt):
    # generation greedy (memes parametres que le benchmark de base) avec mesure de puissance
    inputs = build_inputs(user_prompt)
    n_in = inputs["input_ids"].shape[1]
    torch.cuda.synchronize()
    with Monitor() as mon:
        t0 = time.time()
        out = model.generate(**inputs, max_new_tokens=GEN_PARAMS["max_new_tokens"],
                             do_sample=GEN_PARAMS["do_sample"])
        torch.cuda.synchronize()
        dur = time.time() - t0
    new = out[0][n_in:]
    avg_w = mon.avg_power_w
    return {
        "response": tokenizer.decode(new, skip_special_tokens=True),
        "latency_s": round(dur, 3), "output_tokens": len(new),
        "avg_power_w": round(avg_w, 1), "peak_power_w": round(mon.peak_power_w, 1),
        "energy_wh": round(avg_w * dur / 3600, 5),
        "cpu_rss_peak_go": round(mon.cpu_rss_peak_go, 2),
    }

# Echauffement + controle
test = run_measured("Write a Python function add(a, b) that returns a + b.")
assert "def" in test["response"], "Reponse incoherente : verifiez la version de transformers."
print("Modele pret. Test :", {k: v for k, v in test.items() if k != "response"})

# Consommation au repos (GPU charge mais inactif)
torch.cuda.synchronize()
with Monitor() as idle:
    time.sleep(MON["idle_seconds"])
IDLE_W = idle.avg_power_w
print(f"Puissance au repos : {IDLE_W:.1f} W")

config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/2.93k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 15.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Precision : bf16
Modele pret. Test : {'latency_s': 2.842, 'output_tokens': 36, 'avg_power_w': 52.7, 'peak_power_w': 72.7, 'energy_wh': 0.04159, 'cpu_rss_peak_go': 2.01}
Puissance au repos : 32.4 W


## Partie 1 : benchmark mesuré (15 prompts de base + 9 prompts de sécurité)

In [6]:
base_prompts = json.load(open("../prompts.json"))["prompts"]
rows, vuln_rows = [], []

for group, plist in [("base", base_prompts), ("securite", SEC_PROMPTS)]:
    for p in plist:
        r = run_measured(p["prompt"])
        resp = r.pop("response")
        r["energy_net_wh"] = round(max(0.0, (r["avg_power_w"] - IDLE_W) * r["latency_s"] / 3600), 5)
        r["tokens_per_sec"] = round(r["output_tokens"] / r["latency_s"], 2) if r["latency_s"] > 0 else None
        rows.append({"id": p["id"], "task": p["task"], **r})
        if group == "securite":
            kw = [k.lower() for k in p.get("keywords", [])]
            vuln_rows.append({
                "id": p["id"], "prompt": p["prompt"], "response": resp,
                "has_vuln": p["has_vuln"], "expected_vuln": p["expected_vuln"],
                "mot_cle_trouve": any(k in resp.lower() for k in kw) if kw else None,
                "vuln_label": None,    # a remplir a la main (partie 3)
            })
        print(p["id"], f'{r["latency_s"]}s', f'{r["energy_wh"]} Wh')

df_energy = pd.DataFrame(rows)
df_energy.to_csv(f"../results/energie_{model_slug}.csv", index=False)
pd.DataFrame(vuln_rows).to_csv(f"../results/vulnerabilites_{model_slug}.csv", index=False)
print(f"-> results/energie_{model_slug}.csv, results/vulnerabilites_{model_slug}.csv")
df_energy[["id", "latency_s", "avg_power_w", "energy_wh", "energy_net_wh", "cpu_rss_peak_go"]]

P1 2.758s 0.05001 Wh
P2 6.127s 0.11718 Wh
P3 3.61s 0.06996 Wh
P4 4.396s 0.08431 Wh
P5 1.513s 0.02848 Wh
P6 6.7s 0.12699 Wh
P7 5.744s 0.1082 Wh
P8 7.169s 0.13634 Wh
P9 5.813s 0.1092 Wh
P10 1.375s 0.02641 Wh
P11 7.011s 0.13223 Wh
P12 5.263s 0.09804 Wh
P13 10.19s 0.19075 Wh
P14 10.079s 0.18787 Wh
P15 21.647s 0.40074 Wh
V1 13.816s 0.25731 Wh
V2 15.989s 0.29798 Wh
V3 21.423s 0.39991 Wh
V4 12.389s 0.23108 Wh
V5 12.917s 0.24034 Wh
V6 14.204s 0.26413 Wh
V7 13.619s 0.25481 Wh
V8 23.976s 0.44655 Wh
V9 14.611s 0.27161 Wh
-> results/energie_phi-4-mini-instruct.csv, results/vulnerabilites_phi-4-mini-instruct.csv


,id,latency_s,avg_power_w,energy_wh,energy_net_wh,cpu_rss_peak_go
0,P1,2.758,65.3,0.05001,0.02520,2.01
1,P2,6.127,68.9,0.11718,0.06210,2.01
2,P3,3.610,69.8,0.06996,0.03749,2.01
3,P4,4.396,69.0,0.08431,0.04468,2.01
4,P5,1.513,67.8,0.02848,0.01487,2.01
5,P6,6.700,68.2,0.12699,0.06661,2.01
6,P7,5.744,67.8,0.10820,0.05646,2.01
7,P8,7.169,68.5,0.13634,0.07187,2.01
8,P9,5.813,67.6,0.10920,0.05682,2.01
9,P10,1.375,69.2,0.02641,0.01405,2.01


## Partie 2 : pass@k sur HumanEval (échantillonnage)
Pour chaque problème : `n` réponses par échantillonnage (température, top_p, top_k, graine dans `config_complement.json`), puis estimateur non biaisé de pass@k : `1 - C(n-c, k) / C(n, k)` avec `c` = nombre de réponses correctes.
Les réponses d'un problème sont générées en un seul lot : pas de mesure de latence ici, seulement la justesse.

In [7]:
def pass_at_k(n, c, k):
    if n - c < k:
        return 1.0
    return 1.0 - comb(n - c, k) / comb(n, k)

def sample_n(user_prompt, n):
    inputs = build_inputs(user_prompt)
    n_in = inputs["input_ids"].shape[1]
    out = model.generate(**inputs, max_new_tokens=GEN_PARAMS["max_new_tokens"], do_sample=True,
                         temperature=PK["temperature"], top_p=PK["top_p"], top_k=PK["top_k"],
                         num_return_sequences=n)
    return [tokenizer.decode(o[n_in:], skip_special_tokens=True) for o in out]

subset = json.load(open("../humaneval_subset.json"))
humaneval = load_dataset(HE_CONFIG["dataset"], split="test")
humaneval = humaneval.filter(lambda ex: ex["task_id"] in subset["task_ids"])
HEADER = "from typing import *\nimport math\n\n"
N = PK["n_samples"]

per_problem = []
for i, ex in enumerate(humaneval):
    torch.manual_seed(PK["seed"] + i)
    t0 = time.time()
    samples = sample_n(HE_CONFIG["instruction"] + ex["prompt"], N)
    test_code = ex["test"] + f"\ncheck({ex['entry_point']})"
    c = 0
    for s in samples:
        st, _ = run_tests(HEADER + ex["prompt"] + "\n" + extract_code(s), test_code,
                          timeout=HE_CONFIG["timeout_seconds"])
        c += (st == "pass")
    per_problem.append({"task_id": ex["task_id"], "n": N, "c": c})
    print(ex["task_id"], f"{c}/{N} corrects", f"({time.time() - t0:.0f}s)")

pass_k = {str(k): round(100 * sum(pass_at_k(p["n"], p["c"], k) for p in per_problem) / len(per_problem), 1)
          for k in PK["k_values"]}
summary = {"model": MODEL_NAME, "gpu": GPU_NAME, "precision": config["precision"],
           "n_problems": len(per_problem), "n_samples": N, "temperature": PK["temperature"],
           "top_p": PK["top_p"], "top_k": PK["top_k"], "seed": PK["seed"],
           "pass_at_k": pass_k, "per_problem": per_problem}
with open(f"../results/passk_{model_slug}.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print("pass@k (%) :", pass_k, f"-> results/passk_{model_slug}.json")

README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 83.9kB            

openai_humaneval/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Filter:   0%|          | 0/164 [00:00<?, ? examples/s]

HumanEval/0 10/10 corrects (46s)
HumanEval/1 6/10 corrects (70s)
HumanEval/2 10/10 corrects (30s)
HumanEval/3 10/10 corrects (40s)
HumanEval/4 10/10 corrects (46s)
HumanEval/5 8/10 corrects (39s)
HumanEval/6 10/10 corrects (56s)
HumanEval/7 10/10 corrects (28s)
HumanEval/8 10/10 corrects (40s)
HumanEval/9 10/10 corrects (43s)
HumanEval/10 5/10 corrects (65s)
HumanEval/11 10/10 corrects (36s)
HumanEval/12 10/10 corrects (39s)
HumanEval/13 10/10 corrects (24s)
HumanEval/14 10/10 corrects (23s)
HumanEval/15 10/10 corrects (22s)
HumanEval/16 10/10 corrects (21s)
HumanEval/17 3/10 corrects (87s)
HumanEval/18 10/10 corrects (40s)
HumanEval/19 10/10 corrects (74s)
pass@k (%) : {'1': 91.0, '5': 99.6, '10': 100.0} -> results/passk_phi-4-mini-instruct.json


## Partie 3 : notation manuelle des vulnérabilités
Même logique que les bugs : `TP` (vraie faille ET bon type de faille identifié), `FP` (pas de faille, ou pas celle trouvée, mais le modèle en invente une), `FN` (vraie faille non trouvée), `TN` (pas de faille et le modèle le dit).
La colonne `mot_cle_trouve` n'est qu'une **indication** : un modèle peut citer « injection SQL » pour dire qu'il n'y en a pas. Lisez les réponses.

In [8]:
dfv = pd.read_csv(f"../results/vulnerabilites_{model_slug}.csv")
for _, r in dfv.iterrows():
    print("=" * 90)
    print(r["id"], "| faille reelle :", r["has_vuln"], "|", r["expected_vuln"], "| mot-cle :", r["mot_cle_trouve"])
    print("--- PROMPT :"); print(r["prompt"])
    print("--- REPONSE :"); print(r["response"])

V1 | faille reelle : True | Injection SQL (requete construite par concatenation avec une entree non filtree) | mot-cle : True
--- PROMPT :
Ce code a-t-il une faille de sécurité ? Si oui, laquelle ?

def get_user(conn, username):
    query = "SELECT * FROM users WHERE name = '" + username + "'"
    return conn.execute(query).fetchall()
--- REPONSE :
Oui, le code fourni présente une faille de sécurité connue sous le nom d'injection SQL. Cette vulnérabilité se produit lorsque des entrées utilisateur non validées ou nettoyées sont directement incluses dans une requête SQL, ce qui peut permettre à un attaquant d'exécuter des commandes arbitraires sur la base de données.

Dans ce cas, l'utilisateur de la fonction `get_user` est directement concaténé à la requête SQL sans aucune validation ou nettoyage, ce qui peut permettre à un attaquant de manipuler la requête pour exécuter des commandes SQL malveillantes.

Pour corriger cette failue, vous pouvez utiliser une approche de requête préparée o

In [10]:
# --- A REMPLIR A LA MAIN : "TP", "FP", "FN" ou "TN" pour chaque prompt ---
VULN_LABELS = {f"V{i}": None for i in range(1, 10)}
# -------------------------------------------------------------------------

dfv["vuln_label"] = dfv["vuln_label"].astype(object)
for pid, val in VULN_LABELS.items():
    if val is None:
        continue
    assert val in {"TP", "FP", "FN", "TN"}, f"{pid} : valeur invalide {val!r}"
    assert pid in set(dfv["id"]), f"{pid} inconnu"
    dfv.loc[dfv["id"] == pid, "vuln_label"] = val
dfv.to_csv(f"../results/vulnerabilites_{model_slug}.csv", index=False)
print("Etiquettes encore vides :", int(dfv["vuln_label"].isna().sum()), "/", len(dfv))

Etiquettes encore vides : 9 / 9


## Récupérer vos fichiers

In [11]:
import zipfile
names = [f"results/energie_{model_slug}.csv", f"results/vulnerabilites_{model_slug}.csv",
         f"results/passk_{model_slug}.json"]
out = f"/kaggle/working/complement_{model_slug}.zip" if os.path.exists("/kaggle/working") else f"complement_{model_slug}.zip"
with zipfile.ZipFile(out, "w") as z:
    for n in names:
        z.write("../" + n, arcname=n)
print("Cree :", out)

Cree : /kaggle/working/complement_phi-4-mini-instruct.zip
